In [0]:
CREATE OR REFRESH STREAMING TABLE techdados.bronze.dlt_customers
(
CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL)
)
COMMENT "Raw orders from cloud storage"
AS
SELECT 
    *,
    current_timestamp() AS ingestion_time,
    _metadata.file_path AS source_file
FROM STREAM cloud_files(
    '/Volumes/techdados/bronze/vol_landing/customer',
    'json',
    map(
        'multiLine', 'true',
        'pathGlobFilter', '*.json'
    )
);

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customers_base
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_email EXPECT (email IS NOT NULL AND email LIKE '%@%.%') ON VIOLATION DROP ROW
)
COMMENT 'Silver – customers base'
TBLPROPERTIES ("quality" = "silver")
AS
SELECT
    customer_id,
    name,
    email,
    CAST(created_at AS DATE) AS created_at,
    ingestion_time,
    source_file
FROM STREAM(LIVE.dlt_customers)
WHERE customer_id IS NOT NULL;

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customer_addresses
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_address_type EXPECT (address_type IN ('home', 'work', 'delivery', 'other')) ON VIOLATION DROP ROW,
  CONSTRAINT valid_state EXPECT (LENGTH(state) = 2) ON VIOLATION DROP ROW
)
COMMENT 'Silver – exploded customer addresses with quality checks'
TBLPROPERTIES (
  "quality" = "silver",
  "pipelines.autoOptimize.zOrderCols" = "customer_id"
)
AS
SELECT
    customer_id,
    addr.col.type   AS address_type,
    addr.col.city   AS city,
    addr.col.state  AS state,
    addr.col.zip    AS zip,
    ingestion_time
FROM STREAM(LIVE.dlt_customers)
LATERAL VIEW explode(from_json(addresses, 'array<struct<type:string,city:string,state:string,zip:string>>')) addr

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customer_phones
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_phone_number EXPECT (phone_number IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT phone_not_empty EXPECT (LENGTH(TRIM(phone_number)) > 0) ON VIOLATION DROP ROW
)
COMMENT 'Silver – exploded customer phones with data quality constraints'
TBLPROPERTIES (
  "quality" = "silver",
  "pipelines.autoOptimize.zOrderCols" = "customer_id"
)
AS
SELECT
    customer_id,
    phone.col AS phone_number,
    ingestion_time
FROM STREAM(LIVE.dlt_customers)
LATERAL VIEW explode(from_json(phones, 'array<string>')) phone

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customer_preferences
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_category EXPECT (category IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT category_not_empty EXPECT (LENGTH(TRIM(category)) > 0) ON VIOLATION DROP ROW,
  CONSTRAINT valid_newsletter EXPECT (newsletter IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_sms EXPECT (sms IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Silver – exploded customer preferences with data quality constraints'
AS
SELECT
    customer_id,
    pref.newsletter AS newsletter,
    pref.sms        AS sms,
    category.col    AS category,
    ingestion_time
FROM (
    SELECT 
        customer_id,
        from_json(preferences, 'struct<newsletter:boolean,sms:boolean,categories:array<string>>') AS pref,
        ingestion_time
    FROM STREAM(LIVE.dlt_customers)
)
LATERAL VIEW explode(pref.categories) category

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customer_purchases
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_purchase_value EXPECT (purchase_value IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT positive_purchase_value EXPECT (purchase_value > 0) ON VIOLATION DROP ROW
)
COMMENT 'Silver – exploded customer purchase history with data quality constraints'
AS
SELECT
    customer_id,
    purchase.col AS purchase_value,
    ingestion_time
FROM STREAM(LIVE.dlt_customers)
LATERAL VIEW explode(from_json(purchase_history, 'array<double>')) purchase